In [1]:
import numpy as np
import pandas as pd
import joblib

In [2]:
test = pd.read_csv("../data/test.csv")

In [3]:
test.head(10)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
5,1466,60,RL,75.0,10000,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
6,1467,20,RL,NaN,7980,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,GdPrv,Shed,500,3,2010,WD,Normal
7,1468,60,RL,63.0,8402,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
8,1469,20,RL,85.0,10176,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2010,WD,Normal
9,1470,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,4,2010,WD,Normal


In [4]:
test = test.drop(columns=["PoolQC", "MiscFeature", "Alley", "Fence"], errors="ignore")
cat_col = test.select_dtypes(include="object").columns
for col in cat_col:
    test[col] = test[col].fillna("None")
num_col = test.select_dtypes(include=["int64", "float64"]).columns
for col in num_col:
    test[col] = test[col].fillna(test[col].median())


filling the Nan values in test file

In [5]:
test = pd.get_dummies(test, drop_first=True)

one hot encoding the categorical columns

In [6]:
model = joblib.load("../models/final_house_price_model.pkl")

In [9]:
train_cols = model.feature_names_in_

In [10]:
test = test.reindex(columns=train_cols, fill_value=0)

using only those colummns from test file which are used to train the model

In [11]:
predictions = model.predict(test)

In [12]:
predictions = np.expm1(predictions)

converting to dollar scale

In [15]:
submission = pd.DataFrame({
    "ID": test["Id"],
    "Sale Price $": predictions
})
submission.head(10)

,ID,Sale Price $
0,1461,107463.398144
1,1462,137475.343712
2,1463,162786.483058
3,1464,180199.490762
4,1465,180859.688555
5,1466,155567.619964
6,1467,159147.685763
7,1468,152542.291840
8,1469,163967.364901
9,1470,111932.270700


In [16]:
submission.to_csv("../submission.csv", index=False)